In [1]:
# Version 3: Deep Learning Model for Heart Disease Risk Prediction
# This notebook adds a neural network model using TensorFlow/Keras.

import pandas as pd
import numpy as np
import os
import json
import hashlib
import sys
import shutil
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)


# ------------------------------------------------------------
# Load cleaned heart disease dataset from GitHub
# ------------------------------------------------------------

url = "https://raw.githubusercontent.com/CurlyRivera/blockchain-audit-trail-heart-disease-ml/main/data/heart_disease_clean_v1.csv"

df = pd.read_csv(url)

print("Dataset shape:", df.shape)
df.head()


# ------------------------------------------------------------
# Separate features and target
# ------------------------------------------------------------

X = df.drop(columns=["num", "heart_disease_present"])
y = df["heart_disease_present"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print(y.value_counts())


# ------------------------------------------------------------
# Use the same train/test split method as previous versions
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


# ------------------------------------------------------------
# Scale features for neural network training
# ------------------------------------------------------------

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled training data shape:", X_train_scaled.shape)
print("Scaled testing data shape:", X_test_scaled.shape)


# ------------------------------------------------------------
# Version 3 model type comparison table
# ------------------------------------------------------------

model_type_table = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Gradient Boosting",
        "Neural Network / MLP"
    ],
    "Type": [
        "Traditional ML baseline",
        "Tree-based ML",
        "Boosted tree model",
        "Deep learning baseline"
    ]
})

model_type_table


# ------------------------------------------------------------
# Build neural network / MLP model
# ------------------------------------------------------------

tf.keras.utils.set_random_seed(42)

mlp_model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

mlp_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

mlp_model.summary()


# ------------------------------------------------------------
# Train neural network model
# ------------------------------------------------------------

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = mlp_model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=16,
    callbacks=[early_stopping],
    verbose=1
)


# ------------------------------------------------------------
# Evaluate neural network model
# ------------------------------------------------------------

y_prob_mlp = mlp_model.predict(X_test_scaled).ravel()
y_pred_mlp = (y_prob_mlp >= 0.5).astype(int)

mlp_results = {
    "Model": "Neural Network / MLP",
    "Type": "Deep learning baseline",
    "Accuracy": accuracy_score(y_test, y_pred_mlp),
    "Precision": precision_score(y_test, y_pred_mlp),
    "Recall": recall_score(y_test, y_pred_mlp),
    "F1 Score": f1_score(y_test, y_pred_mlp),
    "ROC-AUC": roc_auc_score(y_test, y_prob_mlp)
}

mlp_results


# ------------------------------------------------------------
# Compare Version 3 neural network against Version 2 models
# ------------------------------------------------------------

version_2_results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Type": "Traditional ML baseline",
        "Accuracy": 0.833,
        "Precision": 0.846,
        "Recall": 0.786,
        "F1 Score": 0.815,
        "ROC-AUC": 0.949
    },
    {
        "Model": "Random Forest",
        "Type": "Tree-based ML",
        "Accuracy": 0.850,
        "Precision": 0.880,
        "Recall": 0.786,
        "F1 Score": 0.830,
        "ROC-AUC": 0.941
    },
    {
        "Model": "Gradient Boosting",
        "Type": "Boosted tree model",
        "Accuracy": 0.767,
        "Precision": 0.769,
        "Recall": 0.714,
        "F1 Score": 0.741,
        "ROC-AUC": 0.883
    }
])

mlp_results_df = pd.DataFrame([mlp_results])

model_comparison_v3 = pd.concat(
    [version_2_results, mlp_results_df],
    ignore_index=True
)

model_comparison_v3_rounded = model_comparison_v3.copy()

for column in ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]:
    model_comparison_v3_rounded[column] = model_comparison_v3_rounded[column].apply(lambda x: f"{x:.1%}")

model_comparison_v3_rounded


# ------------------------------------------------------------
# Save Version 3 model comparison results
# ------------------------------------------------------------

os.makedirs("results", exist_ok=True)

model_comparison_v3.to_csv("results/model_comparison_v3.csv", index=False)

print("Saved: results/model_comparison_v3.csv")

pd.read_csv("results/model_comparison_v3.csv")


# ------------------------------------------------------------
# Step 4 import test: clone GitHub repo and import src files
# ------------------------------------------------------------

repo_dir = "/content/blockchain-audit-trail-heart-disease-ml"
repo_url = "https://github.com/CurlyRivera/blockchain-audit-trail-heart-disease-ml.git"

# Remove old copy if it exists
if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)

# Clone fresh copy of repo
!git clone {repo_url}

# Add src folder to Python path
src_path = f"{repo_dir}/src"
sys.path.insert(0, src_path)

print("Repo cloned.")
print("Source path added:", src_path)
print("Files in src folder:")
print(os.listdir(src_path))


# ------------------------------------------------------------
# Import reusable Python functions from src/
# ------------------------------------------------------------

from data_processing import load_dataset, prepare_features_and_target, split_data, scale_features
from model_training import build_mlp_model, train_mlp_model
from evaluation import evaluate_keras_model, create_results_table
from audit_trail import hash_block, create_audit_block
from tamper_detection import verify_audit_chain
from feature_importance import create_feature_importance_table

print("All src files imported successfully.")

# ------------------------------------------------------------
# Step 6: Create Version 3 audit chain with Neural Network block
# ------------------------------------------------------------

import json
import hashlib
from datetime import datetime

# Save a local copy of the cleaned dataset so it can be hashed
dataset_filename = "heart_disease_clean_v1.csv"
df.to_csv(dataset_filename, index=False)


def hash_file(filename):
    """Create a SHA-256 hash for a file."""
    sha256_hash = hashlib.sha256()

    with open(filename, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)

    return sha256_hash.hexdigest()


def hash_block(block):
    """Create a SHA-256 hash for an audit block."""
    block_string = json.dumps(block, sort_keys=True).encode()
    return hashlib.sha256(block_string).hexdigest()


dataset_hash = hash_file(dataset_filename)

print("Dataset hash:")
print(dataset_hash)


# Create a clean metrics dictionary for each model
def get_model_metrics(model_name):
    """Get performance metrics from model_comparison_v3."""
    row = model_comparison_v3[model_comparison_v3["Model"] == model_name].iloc[0]

    return {
        "Accuracy": float(row["Accuracy"]),
        "Precision": float(row["Precision"]),
        "Recall": float(row["Recall"]),
        "F1 Score": float(row["F1 Score"]),
        "ROC-AUC": float(row["ROC-AUC"])
    }


# Neural network architecture details
mlp_architecture_details = {
    "architecture": "Multilayer Perceptron",
    "number_of_layers": 3,
    "layers": [
        {
            "layer_type": "Dense",
            "units": 16,
            "activation_function": "relu"
        },
        {
            "layer_type": "Dense",
            "units": 8,
            "activation_function": "relu"
        },
        {
            "layer_type": "Dense",
            "units": 1,
            "activation_function": "sigmoid"
        }
    ],
    "optimizer": "adam",
    "learning_rate": float(tf.keras.backend.get_value(mlp_model.optimizer.learning_rate)),
    "epochs_planned": 100,
    "epochs_trained": len(history.history["loss"]),
    "batch_size": 16,
    "loss_function": "binary_crossentropy",
    "early_stopping": {
        "monitor": "val_loss",
        "patience": 10,
        "restore_best_weights": True
    }
}


# Define all Version 3 audit blocks
audit_models_v3 = [
    {
        "block_number": 1,
        "model_name": "Logistic Regression",
        "model_type": "Traditional ML baseline",
        "model_version": "v3_model_1",
        "hyperparameters": {
            "max_iter": 1000
        },
        "deep_learning_details": None
    },
    {
        "block_number": 2,
        "model_name": "Random Forest",
        "model_type": "Tree-based ML",
        "model_version": "v3_model_2",
        "hyperparameters": {
            "random_state": 42
        },
        "deep_learning_details": None
    },
    {
        "block_number": 3,
        "model_name": "Gradient Boosting",
        "model_type": "Boosted tree model",
        "model_version": "v3_model_3",
        "hyperparameters": {
            "random_state": 42
        },
        "deep_learning_details": None
    },
    {
        "block_number": 4,
        "model_name": "Neural Network / MLP",
        "model_type": "Deep learning baseline",
        "model_version": "v3_model_4",
        "hyperparameters": {
            "optimizer": "adam",
            "learning_rate": float(tf.keras.backend.get_value(mlp_model.optimizer.learning_rate)),
            "epochs_planned": 100,
            "epochs_trained": len(history.history["loss"]),
            "batch_size": 16,
            "loss_function": "binary_crossentropy"
        },
        "deep_learning_details": mlp_architecture_details
    }
]


# Build the Version 3 audit chain
audit_chain_v3 = []
previous_block_hash = "0"
run_timestamp = datetime.now().isoformat()

for model_info in audit_models_v3:
    model_name = model_info["model_name"]

    audit_block = {
        "block_number": model_info["block_number"],
        "project_title": "Designing a Blockchain-Inspired Audit Trail for Machine Learning Models Used in Heart Disease Risk Prediction",
        "project_version": "Version 3",
        "dataset_name": "UCI Heart Disease Dataset",
        "dataset_version": "heart_disease_clean_v1",
        "dataset_hash": dataset_hash,
        "target_variable": "heart_disease_present",
        "features_used": list(X.columns),
        "train_test_split": {
            "test_size": 0.2,
            "random_state": 42,
            "stratified": True
        },
        "model_name": model_name,
        "model_type": model_info["model_type"],
        "model_version": model_info["model_version"],
        "training_date": run_timestamp,
        "hyperparameters": model_info["hyperparameters"],
        "deep_learning_details": model_info["deep_learning_details"],
        "performance_metrics": get_model_metrics(model_name),
        "previous_block_hash": previous_block_hash
    }

    audit_block["current_block_hash"] = hash_block(audit_block)
    audit_chain_v3.append(audit_block)

    previous_block_hash = audit_block["current_block_hash"]


# Save Version 3 audit chain
os.makedirs("results", exist_ok=True)

with open("results/audit_chain_v3.json", "w") as f:
    json.dump(audit_chain_v3, f, indent=4)

print("Saved: results/audit_chain_v3.json")
print(f"Total audit blocks created: {len(audit_chain_v3)}")


# Display audit chain summary
for block in audit_chain_v3:
    print("Block Number:", block["block_number"])
    print("Model Name:", block["model_name"])
    print("Model Version:", block["model_version"])
    print("Previous Block Hash:", block["previous_block_hash"])
    print("Current Block Hash:", block["current_block_hash"])
    print("-" * 80)

TensorFlow version: 2.20.0
Dataset shape: (297, 15)
Feature matrix shape: (297, 13)
Target shape: (297,)
heart_disease_present
0    160
1    137
Name: count, dtype: int64
X_train: (237, 13)
X_test: (60, 13)
y_train: (237,)
y_test: (60,)
Scaled training data shape: (237, 13)
Scaled testing data shape: (60, 13)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 369 (1.44 KB)

 Trainable params: 369 (1.44 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 4s 95ms/step - accuracy: 0.3439 - loss: 0.7951 - val_accuracy: 0.4583 - val_loss: 0.7200
Epoch 2/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.4074 - loss: 0.7475 - val_accuracy: 0.5000 - val_loss: 0.6971
Epoch 3/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.4868 - loss: 0.7066 - val_accuracy: 0.5417 - val_loss: 0.6790
Epoch 4/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.5661 - loss: 0.6703 - val_accuracy: 0.6458 - val_loss: 0.6645
Epoch 5/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.6085 - loss: 0.6375 - val_accuracy: 0.6458 - val_loss: 0.6523
Epoch 6/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.6772 - loss: 0.6091 - val_accuracy: 0.6458 - val_loss: 0.6421
Epoch 7/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7090 - loss: 0.5841 - val_accuracy: 0.6458 - val_loss: 0.6338
Epoch 8/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.7302 - loss: 0.5618 - val_accuracy: 0.